# Marcus theory of electron transfer and the inverted region

*physchem · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/marcus_electron_transfer)

An electron jumps from a donor to an acceptor only when the surrounding solvent happens to fluctuate into a configuration where both states have the same energy: the crossing of two parabolas. That gives the Marcus rate $k \propto \exp[-(\Delta G^\circ + \lambda)^2/4\lambda k_BT]$, evaluated here on one broadcast grid of reorganisation energies, temperatures and driving forces. It is fastest when $-\Delta G^\circ = \lambda$ and, against intuition, slows down again for even more downhill reactions. A Langevin simulation of the solvent coordinate with Landau–Zener hops at the crossing reproduces the curve.

**Vector form**

$$k = \frac{2\pi}{\hbar}\,|H_{AB}|^2\,\frac{1}{\sqrt{4\pi\lambda k_BT}}\,\exp\!\Big[-\frac{(\Delta G^\circ + \lambda)^2}{4\lambda k_BT}\Big],\qquad G_R(x) = \lambda x^2,\quad G_P(x) = \lambda(x-1)^2 + \Delta G^\circ$$

**Index (tensor) form**

$$k_{ltg} = \frac{2\pi}{\hbar}\,|H_{AB}|^2\,(4\pi\lambda_l k_BT_t)^{-1/2}\exp\!\Big[-\frac{(\Delta G^\circ_g + \lambda_l)^2}{4\lambda_l k_BT_t}\Big] \qquad \ddot x_{dn} = -\omega^2(x_{dn} - s_{dn}) - \gamma\dot x_{dn} + \sqrt{2\gamma\,\omega^2 k_BT/2\lambda}\;\xi_{dn} \qquad P^{\mathrm{LZ}}_{dn} = 1 - \exp\!\Big[-\frac{2\pi|H_{AB}|^2}{\hbar\,|\dot x_{dn}|\,2\lambda}\Big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\Delta G^\circ_g,\ \lambda_l,\ T_t$ | `driving_forces(...), reorganisation_energies(lam), temperatures(T)` | (G,), (L,), (T,) | the three axes of the rate grid |
| $k_{ltg}$ | `marcus_rate(dG[None, None, :], lam[:, None, None], kT[None, :, None], H)` | (L, T, G) | broadcasting builds every combination at once; no index is summed |
| $x^* = \frac{\lambda + \Delta G^\circ}{2\lambda}$ | `crossing_point(lam, dG)` | (D,) | where the parabolas cross; $x^* < 0$ means the inverted region |
| $\Delta G^\ddagger = \frac{(\Delta G^\circ + \lambda)^2}{4\lambda}$ | `activation_energy(lam, dG)` | (D,) | height of the crossing above the reactant minimum |
| $x_{dn},\ \dot x_{dn}$ | `thermal_ensemble(D, N, lam, kT, omega)` | (D, N) | trajectory $n$ at simulated driving force $d$, started in the reactant well with Boltzmann statistics |
| $s_{dn}$ | `state` | (D, N) | 0 on the reactant surface, 1 on the product surface; it sets the minimum the solvent is pulled to |
| $\text{BAOAB}$ | `hopping_dynamics(x, v, dG, ...)` | (D, N) | half kick, half drift, friction + noise, half drift, half kick for all trajectories together |
| $P^{\mathrm{LZ}}_{dn}$ | `landau_zener(abs(x - x_old) / dt, lam, H)` | (D, N) | switch surface with this probability whenever a step passes $x^*$ |
| $k^{\text{sim}}_d$ | `hops / t_react` | (D,) | forward hops per unit time spent on the reactant surface; error bar $\sqrt{\text{hops}}/t$ |

Units: eV, K and ps. The solvent coordinate $x$ is the collective polarisation that the transferred charge sees, scaled so the reactant well sits at 0 and the product well at 1; $\lambda$ is the energy it costs to polarise the solvent as for the product while the charge is still on the donor. Marcus derived the quadratic barrier in 1956 (J. Chem. Phys. 24, 966) and predicted the inverted region, which was confirmed experimentally only in the 1980s. In the inverted region ($-\Delta G^\circ > \lambda$) the product parabola crosses the reactant one on its far side, so a larger driving force pushes the crossing higher. The rate formula is the golden rule (non-adiabatic, small $H_{AB}$) with a classical solvent; real inverted-region rates fall more slowly because high-frequency molecular vibrations open quantum channels (the Marcus–Levich–Jortner extension). The simulation is a deliberately simple picture: one classical solvent coordinate with frequency $\omega$ and friction $\gamma$, moving on one diabatic parabola at a time and switching at the crossing with the Landau–Zener probability $1 - e^{-2\pi H_{AB}^2/\hbar|\dot x|\,|\Delta F|}$, which assumes a straight, fast passage. Summed over a thermal flux of passages that gives back the golden-rule rate when $P_{\mathrm{LZ}} \ll 1$, which is why the points land on the curve; for large $H_{AB}$ the hops saturate (adiabatic transfer) and friction starts to matter. Trajectories that hop continue on the product surface and may hop back. Try lowering $T$ (the curve narrows), raising $H_{AB}$ (the simulation drifts below the formula) or changing $\lambda$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/marcus_electron_transfer/main.py
"""Main loop: Marcus rate curves for a batch of λ and T → surface-hopping simulation → compare."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import activation_energy, crossing_point, marcus_rate


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam, T, H_ab = params["lam"], params["T"], params["H_ab"]
    kT = tensors.KB * T
    dG = tensors.driving_forces(-3 * lam, lam, 241, dev)  # (G,)
    lams = tensors.reorganisation_energies(lam, dev)  # (L,)
    temps = tensors.temperatures(T, dev)  # (T,)
    k = solve.rate_grid(dG, lams, tensors.KB * temps, H_ab)  # (L, T, G)
    print(f"Marcus rate on a grid of {len(lams)} λ × {len(temps)} T × {len(dG)} ΔG° = {tuple(k.shape)}")
    for i, lm in enumerate(lams.tolist()):
        g = int(k[i, 1].argmax())
        print(f"  λ = {lm:.2f} eV: fastest at ΔG° = {dG[g].item():+.3f} eV (expected -λ), k_max = {k[i, 1, g]:.3g} /ps")

    # stochastic check: Langevin motion of the solvent + Landau–Zener hops, one row per driving force
    omega, gamma = params["omega"], params["friction"]
    dt = 0.1 / omega  # ten steps per radian of solvent motion
    n_steps = 300 if fast_mode() else params["steps"]
    n_traj = 400 if fast_mode() else 2000
    dG_sim = tensors.simulated_driving_forces(lam, 9, dev)  # (D,)
    x, v = tensors.thermal_ensemble(len(dG_sim), n_traj, lam, kT, omega, dev)
    snaps = solve.snapshot_steps(n_steps, 6)
    threads = torch.get_num_threads()
    torch.set_num_threads(1)  # tensors of 10⁴ numbers: starting threads every step costs more than it saves
    try:
        res = solve.hopping_dynamics(x, v, dG_sim, lam, kT, H_ab, omega, gamma, dt, n_steps, snaps)
    finally:
        torch.set_num_threads(threads)
    k_sim, k_err = solve.simulated_rates(res)
    k_th = marcus_rate(dG_sim, torch.tensor(lam), torch.tensor(kT), H_ab)
    print(
        f"\nsurface hopping: {len(dG_sim)} ΔG° × {n_traj} trajectories, {n_steps} steps × dt = {dt * 1e3:.2f} fs "
        f"= {n_steps * dt:.1f} ps, k_BT = {kT * 1e3:.1f} meV"
    )
    print("   ΔG° (eV)   x*      ΔG‡ (eV)   hops   P_LZ    k_sim (1/ps)        k_Marcus (1/ps)")
    for d in range(len(dG_sim)):
        g0 = dG_sim[d].item()
        print(
            f"   {g0:+.3f}   {crossing_point(lam, g0):+.3f}   {activation_energy(lam, g0):.3f}     "
            f"{int(res['hops'][d]):5d}  {res['p_lz'][d]:.3f}   {k_sim[d]:.3g} ± {k_err[d]:.2g}      {k_th[d]:.3g}"
        )
    ok = (res["hops"] > 0) & (k_th > 0)
    ratio = (k_sim[ok] / k_th[ok]).log10().abs().max().item()
    print(f"largest |log10(k_sim / k_Marcus)| = {ratio:.2f}  (Landau–Zener hops give the golden rule when P_LZ ≪ 1)")
    return plot.animate(dG, lams, temps, k, dG_sim, k_sim, k_err, res, snaps, dt, lam, T, out_dir / "marcus.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/marcus_electron_transfer/equation.py
r"""Physics → tensor translation: Marcus parabolas, the Marcus rate and the Landau–Zener hopping probability.

Indices: l reorganisation energy, t temperature, g driving force, d simulated driving force, n trajectory.
No index is summed here: every formula is evaluated on a broadcast grid, e.g. k_ltg for all λ_l, T_t, ΔG°_g at once.

    G_R(x) = λ x²,   G_P(x) = λ (x - 1)² + ΔG°                     free-energy surfaces of the solvent coordinate
    x* = (λ + ΔG°) / 2λ,   ΔG‡ = G_R(x*) = (ΔG° + λ)² / 4λ           where the parabolas cross
    k_ltg = (2π/ħ) |H_AB|² (4π λ_l k_B T_t)^{-1/2} exp(-(ΔG°_g + λ_l)² / 4 λ_l k_B T_t)
    P_LZ = 1 - exp(-2π |H_AB|² / (ħ |ẋ| |G_P' - G_R'|)),  |G_P' - G_R'| = 2λ
"""

import math

import torch

from .tensors import HBAR


def reactant_surface(x: torch.Tensor, lam) -> torch.Tensor:
    """G_R(x) = λ x² (eV)."""
    return lam * x**2


def product_surface(x: torch.Tensor, lam, dG) -> torch.Tensor:
    """G_P(x) = λ (x - 1)² + ΔG° (eV)."""
    return lam * (x - 1) ** 2 + dG


def crossing_point(lam, dG):
    """x* where G_R = G_P: λx² = λ(x-1)² + ΔG°  ⇒  x* = (λ + ΔG°) / 2λ."""
    return (lam + dG) / (2 * lam)


def activation_energy(lam, dG):
    """ΔG‡ = G_R(x*) = (ΔG° + λ)² / 4λ: zero when -ΔG° = λ, rising again beyond (the inverted region)."""
    return (dG + lam) ** 2 / (4 * lam)


def marcus_rate(dG: torch.Tensor, lam: torch.Tensor, kT: torch.Tensor, H_ab: float) -> torch.Tensor:
    """Non-adiabatic Marcus rate (1/ps). Broadcasts: e.g. dG (1, 1, G), lam (L, 1, 1), kT (1, T, 1) → (L, T, G)."""
    prefactor = 2 * math.pi / HBAR * H_ab**2 / torch.sqrt(4 * math.pi * lam * kT)
    return prefactor * torch.exp(-activation_energy(lam, dG) / kT)


def landau_zener(speed: torch.Tensor, lam, H_ab: float) -> torch.Tensor:
    """Probability of switching from one diabatic surface to the other in one passage of the crossing.

    speed |ẋ| in 1/ps; the slopes of the parabolas differ by 2λ everywhere (same curvature).
    """
    return 1 - torch.exp(-2 * math.pi * H_ab**2 / (HBAR * speed.clamp_min(1e-12) * 2 * lam))


def solvent_force(x: torch.Tensor, state: torch.Tensor, omega: float) -> torch.Tensor:
    """Acceleration on the current surface: ẍ = -ω² (x - x_min), x_min = 0 on the reactant, 1 on the product."""
    return -(omega**2) * (x - state)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/marcus_electron_transfer/tensors.py
"""Build the concrete tensors: driving forces, reorganisation energies, temperatures and the starting ensemble.

Units: energies in eV, temperatures in K, time in ps. The solvent coordinate x is dimensionless: the reactant
parabola has its minimum at x = 0 and the product parabola at x = 1.
"""

import torch

KB = 8.617333e-5  # Boltzmann constant, eV/K
HBAR = 6.582120e-4  # reduced Planck constant, eV·ps


def driving_forces(dG_min: float, dG_max: float, n: int, device: torch.device) -> torch.Tensor:
    """ΔG°_g, shape (G,): reaction free energies for the rate curves (negative = downhill)."""
    return torch.linspace(dG_min, dG_max, n, device=device)


def reorganisation_energies(lam: float, device: torch.device) -> torch.Tensor:
    """λ_l, shape (L,): half, once and twice the chosen reorganisation energy."""
    return lam * torch.tensor([0.5, 1.0, 2.0], device=device)


def temperatures(T: float, device: torch.device) -> torch.Tensor:
    """T_t, shape (3,): the chosen temperature and ±100 K around it."""
    return torch.tensor([T - 100.0, T, T + 100.0], device=device).clamp_min(50.0)


def simulated_driving_forces(lam: float, n: int, device: torch.device) -> torch.Tensor:
    """ΔG°_d, shape (D,): from 0 (normal region) through -λ (top of the curve) to -2λ (deep in the inverted region)."""
    return torch.linspace(0.0, -2 * lam, n, device=device)


def thermal_ensemble(n_dG: int, n_traj: int, lam: float, kT: float, omega: float, device: torch.device, seed: int = 0):
    """x, v of shape (D, N): every trajectory starts on the reactant surface, drawn from its Boltzmann distribution.

    G_R = λx² is a harmonic well, so ⟨x²⟩ = k_BT / 2λ and, with frequency ω, ⟨v²⟩ = ω² k_BT / 2λ.
    """
    gen = torch.Generator().manual_seed(seed)
    sx = (kT / (2 * lam)) ** 0.5
    x = sx * torch.randn(n_dG, n_traj, generator=gen)
    v = omega * sx * torch.randn(n_dG, n_traj, generator=gen)
    return x.to(device), v.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/marcus_electron_transfer/solve.py
"""Rate curves on a broadcast grid, and Langevin dynamics with Landau–Zener surface hopping for every ΔG° at once."""

import math

import torch

from problems._common.progress import steps

from .equation import crossing_point, landau_zener, marcus_rate, solvent_force


def rate_grid(dG: torch.Tensor, lam: torch.Tensor, kT: torch.Tensor, H_ab: float) -> torch.Tensor:
    """k_ltg for every reorganisation energy, temperature and driving force.   (L, T, G)"""
    return marcus_rate(dG[None, None, :], lam[:, None, None], kT[None, :, None], H_ab)


def hopping_dynamics(
    x: torch.Tensor,
    v: torch.Tensor,
    dG: torch.Tensor,
    lam: float,
    kT: float,
    H_ab: float,
    omega: float,
    gamma: float,
    dt: float,
    n_steps: int,
    snapshots: list[int],
    seed: int = 1,
) -> dict:
    """Underdamped Langevin motion of the solvent coordinate on the current diabatic surface (BAOAB, mass-weighted).

    x, v (D, N): one row per driving force ΔG°_d (dG (D,)). Each time a trajectory passes x* it switches surface
    with the Landau–Zener probability. The forward rate is (number of R → P hops) / (time spent on R).
    Returns hop counts (D,), time on R (D,) in ps, mean P_LZ per passage (D,), and snapshots (S, D, N) of x and state.
    """
    gen = torch.Generator().manual_seed(seed)
    x_star = crossing_point(lam, dG)[:, None]  # (D, 1)
    state = torch.zeros_like(x)  # 0 = reactant (electron on the donor), 1 = product
    c1 = math.exp(-gamma * dt)
    c2 = math.sqrt((1 - c1**2) * kT / (2 * lam)) * omega  # thermal velocity kick
    hops = torch.zeros(len(dG), device=x.device)
    t_react = torch.zeros(len(dG), device=x.device)
    p_sum = torch.zeros(len(dG), device=x.device)
    n_pass = torch.zeros(len(dG), device=x.device)
    snap_x, snap_s = [], []
    for step in steps(n_steps, "Langevin steps"):
        x_old = x
        t_react += dt * (1 - state).sum(1)
        v = v + 0.5 * dt * solvent_force(x, state, omega)  # B
        x = x + 0.5 * dt * v  # A
        v = c1 * v + c2 * torch.randn(v.shape, generator=gen).to(v.device)  # O
        x = x + 0.5 * dt * v  # A
        # passage through the crossing during this step → Landau–Zener hop
        crossed = (x_old - x_star) * (x - x_star) < 0
        p = landau_zener((x - x_old).abs() / dt, lam, H_ab)
        hop = crossed & (torch.rand(x.shape, generator=gen).to(x.device) < p)
        hops += (hop & (state == 0)).sum(1)
        p_sum += (p * crossed).sum(1)
        n_pass += crossed.sum(1)
        state = torch.where(hop, 1 - state, state)
        v = v + 0.5 * dt * solvent_force(x, state, omega)  # B, on the (possibly new) surface
        if step + 1 in snapshots:
            snap_x.append(x.clone())
            snap_s.append(state.clone())
    return {
        "hops": hops,
        "t_react": t_react,
        "p_lz": p_sum / n_pass.clamp_min(1),
        "snap_x": torch.stack(snap_x),
        "snap_state": torch.stack(snap_s),
    }


def simulated_rates(result: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """k = hops / time on R with a Poisson error bar √hops / time.   (D,), (D,)"""
    k = result["hops"] / result["t_react"]
    return k, result["hops"].sqrt() / result["t_react"]


def snapshot_steps(n_steps: int, n: int) -> list[int]:
    """Up to n steps spread geometrically from step 10 to the last: early times show the hopping start."""
    return sorted({round(10 * (n_steps / 10) ** (k / (n - 1))) for k in range(n)})


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/marcus_electron_transfer/plot.py
"""Animate: trajectories hopping between the two parabolas for each ΔG°, and the rate against driving force."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import crossing_point, product_surface, reactant_surface

REACT, PROD, CROSS = "#2563eb", "#dc2626", "#f59e0b"
LAM_COLOURS = ["#16a34a", "#2563eb", "#9333ea"]
N_SHOWN = 150  # trajectories drawn on the parabolas


def animate(
    dG: torch.Tensor,
    lams: torch.Tensor,
    temps: torch.Tensor,
    k: torch.Tensor,
    dG_sim: torch.Tensor,
    k_sim: torch.Tensor,
    k_err: torch.Tensor,
    res: dict,
    snaps: list[int],
    dt: float,
    lam: float,
    T: float,
    path: Path,
) -> Path:
    """k (L, T, G) on the grid dG (G,); simulation at dG_sim (D,); snapshots res['snap_x'] (S, D, N)."""
    dG, lams, temps, k = (a.cpu().numpy() for a in (dG, lams, temps, k))
    dG_sim, k_sim, k_err = (a.cpu().numpy() for a in (dG_sim, k_sim, k_err))
    sx = res["snap_x"][:, :, :N_SHOWN].cpu().numpy()
    ss = res["snap_state"][:, :, :N_SHOWN].cpu().numpy()
    n_snap, n_d = sx.shape[0], len(dG_sim)

    fig, (ax_g, ax_k) = plt.subplots(1, 2, figsize=(10, 4.0), gridspec_kw={"width_ratios": [1, 1.15]})
    xs = np.linspace(-0.9, 2.2, 300)
    (curve_r,) = ax_g.plot(xs, reactant_surface(xs, lam), color=REACT, lw=2, label="reactant  $G_R = \\lambda x^2$")
    (curve_p,) = ax_g.plot([], [], color=PROD, lw=2, label="product  $G_P = \\lambda(x-1)^2 + \\Delta G°$")
    (cross,) = ax_g.plot([], [], "o", color=CROSS, ms=8, zorder=4)
    dots = ax_g.scatter(np.zeros(N_SHOWN), np.zeros(N_SHOWN), s=10, zorder=3)
    ax_g.set_xlim(xs[0], xs[-1])
    ax_g.set_ylim(-2.2 * lam, 1.6 * lam)
    ax_g.set_xlabel("solvent coordinate x")
    ax_g.set_ylabel("free energy (eV)")
    ax_g.legend(fontsize=7, loc="lower left")
    title_g = ax_g.set_title("", fontsize=10)
    info = ax_g.text(0.98, 0.97, "", transform=ax_g.transAxes, ha="right", va="top", fontsize=8)

    floor = 1e-6 * k.max()
    for i, lm in enumerate(lams):
        ax_k.semilogy(
            -dG, np.maximum(k[i, 1], floor), color=LAM_COLOURS[i], lw=1.8, label=f"λ = {lm:.2f} eV, {T:.0f} K"
        )
    for j in (0, 2):
        ax_k.semilogy(
            -dG,
            np.maximum(k[1, j], floor),
            color=LAM_COLOURS[1],
            lw=1,
            ls=":",
            label=f"λ = {lam:.2f} eV, {temps[j]:.0f} K",
        )
    ax_k.axvline(lam, color="#9ca3af", lw=1, ls="--")
    ax_k.text(lam, floor * 3, " $-\\Delta G° = \\lambda$", fontsize=8, color="#6b7280")
    ax_k.text(0.97, 0.04, "inverted\nregion →", transform=ax_k.transAxes, ha="right", fontsize=8, color="#6b7280")
    pts = ax_k.errorbar([], [], yerr=[], fmt="o", color="#374151", ms=5, label="Langevin + Landau–Zener")
    marker = ax_k.axvline(0, color=CROSS, lw=1.5)
    ax_k.set_xlim(-dG[-1], -dG[0])
    ax_k.set_ylim(floor, 5 * k.max())
    ax_k.set_xlabel("driving force −ΔG° (eV)")
    ax_k.set_ylabel("rate k (1/ps)")
    ax_k.set_title("Marcus rate: rises, peaks at −ΔG° = λ, falls", fontsize=10)
    ax_k.legend(fontsize=7, loc="lower center")
    shown = {"bars": pts}

    def update(f: int):
        d, s = divmod(f, n_snap)
        g0 = dG_sim[d]
        curve_p.set_data(xs, product_surface(xs, lam, g0))
        xc = crossing_point(lam, g0)
        cross.set_data([xc], [reactant_surface(xc, lam)])
        x, st = sx[s, d], ss[s, d]
        y = np.where(st == 0, reactant_surface(x, lam), product_surface(x, lam, g0))
        dots.set_offsets(np.c_[x, y])
        dots.set_color(np.where(st[:, None] == 0, np.array([[0.15, 0.39, 0.92, 1]]), np.array([[0.86, 0.15, 0.15, 1]])))
        title_g.set_text(f"ΔG° = {g0:+.2f} eV,  t = {snaps[s] * dt:.2f} ps")
        info.set_text(f"on product: {st.mean():.0%}")
        shown["bars"].remove()
        shown["bars"] = ax_k.errorbar(
            -dG_sim[: d + 1], k_sim[: d + 1], yerr=k_err[: d + 1], fmt="o", color="#374151", ms=5, capsize=2
        )
        marker.set_xdata([-g0, -g0])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_d * n_snap, 54), path, fps=5)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'lam': 0.5,
    'T': 300.0,
    'H_ab': 0.005,
    'omega': 20.0,
    'friction': 10.0,
    'steps': 4000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.marcus_electron_transfer")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.marcus_electron_transfer.main")
problem_dir = Path("problems/physchem/marcus_electron_transfer")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
